# 28. Ders: Buzdolabı Şefi'ni Web'e Taşımak (Arama Motoru Arayüzü)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Google'a bir kelime yazdığınızda Google sizi siyah bir ekrana atıp "Arama Skoru: 0.85" demez. Size bulduğu sonucu büyük, kalın harflerle başlık olarak, altında da ufak bir açıklama ile sunar. 

12. derste Buzdolabı Şefi (Fridge Chef) projemizin algoritmasını yazmış, TF-IDF ve Kosinüs Benzerliği (Cosine Similarity) ile kullanıcının elindeki malzemelere en uygun tarifi bulmuştuk. Şimdi bu projeyi Gradio'ya taşıyacağız. Ancak bir farkla: Bulduğumuz sonucu kullanıcıya kuru bir metin olarak değil, (HTML adı verilen web biçimlendirme etiketlerini kullanarak) şık, kalın puntolu ve göze hitap eden bir sonuç ekranıyla sunacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Output Formatting (Çıktı Biçimlendirme): Bilgisayarın bulduğu ham verileri (isim, skor) kullanıcının gözüne güzel görünecek şekilde tasarlama işlemi.
- HTML (Hypertext Markup Language): Web sitelerinin iskeletini oluşturan, metinleri kalınlaştırmaya, büyütmeye yarayan işaretleme dili.
- Dataset Search (Veri Setinde Arama): Kullanıcı girdisinin arka planda bir Pandas DataFrame'i taranarak en uygun satırın bulunması.
- Data Privacy (Veri Gizliliği): Tarama işleminin tamamen bellekte yapılıp veritabanı dosyalarının kullanıcıdan gizlenmesi.

## Adım 3: Scikit-Learn ve Pandas ile Altyapıyı Hazırlama

Öncelikle mutfağımızı (veri setimizi) ve TF-IDF aracımızı 12. dersteki gibi koda dahil edip eğitelim.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr

# Tarif veritabanımız (Backend)
tarifler = {
    "Yemek Adı": ["Menemen", "Krep", "Tavuk Sote", "Makarna", "Salata"],
    "Malzemeler": [
        "domates biber yumurta yağ tuz",
        "süt un yumurta tuz yağ",
        "tavuk biber soğan domates yağ baharat",
        "makarna su tuz yağ salça",
        "marul domates salatalık limon zeytinyağı"
    ]
}
df_tarif = pd.DataFrame(tarifler)

# Vektörizasyon (Öğrenme) işlemi - Web sitesi açılmadan önce sistem bunu bir kez öğrenip hafızaya alır
vektor_donusturucu = TfidfVectorizer()
tarif_matrisi = vektor_donusturucu.fit_transform(df_tarif["Malzemeler"])

print("Yemek veritabanı kodlandı ve arama motoru hazır!")

## Adım 4: Benzerlik Bulan Çekirdek Fonksiyonu (HTML Çıktılı)

Şimdi arayüzden gelecek kullanıcı malzemelerini alıp, DataFrame'de arayıp, sonucu HTML olarak (Markdown'a çok benzer) döndüren fonksiyonu yazalım.

In [ ]:
def tarif_bul(kullanici_girdisi):
    # Kullanıcı boş bırakırsa hata vermemesi için
    if len(kullanici_girdisi.strip()) == 0:
        return "<h3>Lütfen buzdolabınızdaki malzemeleri yazın!</h3>"
        
    # 1. Kullanıcı metnini sayılara (vektöre) çevir
    kullanici_matrisi = vektor_donusturucu.transform([kullanici_girdisi])
    
    # 2. Sistemdeki tariflerle çarpıştır (Kosinüs Benzerliği)
    skorlar = cosine_similarity(kullanici_matrisi, tarif_matrisi)[0]
    
    # 3. En yüksek skorlu tarifi bul
    en_iyi_indeks = np.argmax(skorlar)
    en_iyi_skor = skorlar[en_iyi_indeks]
    
    # 4. Sonuçları Çıkar ve HTML/Markdown ile Biçimlendir
    if en_iyi_skor == 0:
        return "<h3>Maalesef bu malzemelerle eşleşen bir tarif bulamadım. 😔</h3>"
    else:
        secilen_yemek = df_tarif["Yemek Adı"].iloc[en_iyi_indeks]
        gerekli_malzemeler = df_tarif["Malzemeler"].iloc[en_iyi_indeks]
        yuzde_uyum = int(en_iyi_skor * 100)
        
        # HTML Tagleri: <h2> (Büyük Başlık), <b> (Kalın Yazı), <br> (Alt Satıra Geç)
        sihirli_cikti = f"""
        <h2>🍳 Şefin Önerisi: {secilen_yemek}</h2>
        <b>Sizinle Uyum Oranı:</b> %{yuzde_uyum} <br><br>
        <b>Bu tarif için gereken tam malzemeler:</b> <i>{gerekli_malzemeler}</i>
        """
        return sihirli_cikti

## Adım 5: Gradio HTML Çıktı Bileşeni ile Arayüzü Kurma

Daha önce çıktı olarak `outputs="text"` (yani `gr.Textbox`) kullanmıştık. Ancak HTML kodlarının (<h2>, <b>) çalışabilmesi için özel bir bileşen olan `gr.HTML` bileşenini kullanacağız.

In [ ]:
arayuz = gr.Interface(
    fn=tarif_bul,
    inputs=gr.Textbox(lines=2, placeholder="Buzdolabındaki malzemeleri yaz (Örn: süt, yumurta, un)", label="Elinizdeki Malzemeler"),
    outputs=gr.HTML(label="Tarif Sonucu"), # HTML olarak yorumlayıp kalın puntolarla çizecek
    title="👨‍🍳 Buzdolabı Şefi (Yapay Zeka Destekli)",
    description="Elinizdeki malzemeleri aralarına boşluk veya virgül koyarak yazın. Yapay zeka sistemimiz size en uygun tarifi çıkarsın."
)

print("Arayüz tasarımı tamamlandı.")

## Adım 6: Uygulamayı Yayınlama (Launch)

Sitemizi çalıştıralım.

In [ ]:
arayuz.launch(share=True)

## Adım 7: Görüntü (Resim) Yükleme Mantığına Kısa Bakış

İleride, kullanıcının yazarak değil de "Buzdolabının fotoğrafını çekip" yükleyeceği bir sistem yapmak isterseniz, Gradio'nun `gr.Image()` girdi bileşenini kullanırsınız. Kullanıcı arayüzdeki kutuya resmi sürükler, Gradio bu resmi okur ve (21. Dersteki gibi) bir sınıflandırma algoritmasına vererek içindeki domatesi/soğanı otomatik bulabilir. Bu, yapay zeka mühendislerinin uçtan uca sistem kurma yolculuğudur.

## Adım 8: Dene ve Keşfet - Görsel HTML Çıktıları

Sitenizdeki çıktı ekranının ne kadar profesyonel göründüğüne dikkat edin. Adım 4'teki `sihirli_cikti` değişkeninin içine girip, kendi yaratıcılığınıza göre renk kodları (Örn: `<span style="color:green">%Uyum</span>`) eklemeyi deneyin ve kodları baştan çalıştırın.

## Adım 9: Sıra Sende Görevi - Sadece Tatlılar Butonu (Checkbox)

Problem: Kullanıcı bazen elindeki un ve sütle (krepe rağmen) sadece tatlı (Örn: Kek) yemek isteyebilir. Arayüze bir Checkbox (Onay kutusu) ekleyerek, sadece tatlıları arayan bir filtre mekanizması kurmanız isteniyor.

Gereksinimler:
1. Adım 3'teki DataFrame'e "Kek" tarifini ekleyin (`"Malzemeler": "un süt şeker yumurta kabartma tozu"`).
2. `tarif_bul` fonksiyonunu 2 parametre alacak şekilde güncelleyin: `def tarif_bul(kullanici_girdisi, sadece_tatli_mi):`
3. İçeride bir `if sadece_tatli_mi == True:` kontrolü yapın. (Gerçek projede bu aşamada veritabanı Pandas ile filtrenebilir. Ancak şimdilik sadece kavramsal olarak bu mantığı kavramanız yeterlidir).
4. Arayüzün `inputs` kısmına bir liste olarak metin kutusunu ve Onay kutusunu ekleyin: 
   `inputs=[gr.Textbox(...), gr.Checkbox(label="Sadece Tatlıları Göster")]`
5. Çıktı yine HTML olsun ve siteyi başlatın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 28. Dersi tamamlayarak, devasa veri tabanlarındaki (DataFrame) benzerlik (Similarity) algoritmalarını kullanıcı dostu şık web arayüzlerine (HTML Formatting) bağlamayı başardınız.

Bir sonraki derste, çok uzun metinleri (Özgeçmişler, Makaleler) kopyalayıp yapıştırarak yapay zekaya (Hugging Face) özetleteceğimiz Bulletproof (Kariyer Terzisi) uygulamasını ayağa kaldıracağız.